## Creating Employee Table and Inserting Values

In [0]:
CREATE TABLE employee_payments (
emp_id INT PRIMARY KEY,
emp_name VARCHAR(50),
department VARCHAR(30),
base_salary DECIMAL(10,2),
bonus DECIMAL(10,2),
joining_date DATE
);

INSERT INTO employee_payments VALUES
(1,'karthik','Data',75000.75,5000.50,'2019-03-15'),
(2,'veena','HR',65000.40,4000.25,'2021-06-20'),
(3,'ravi','Data',85000.90,6000.75,'2016-01-10'),
(4,'anil','Finance',70000.10,NULL,'2020-09-01'),
(5,'suresh','HR',60000.55,3000.30,'2022-11-25');


num_affected_rows,num_inserted_rows
5,5


### · Convert emp_name to proper case ---upper /lower ---Initcap (CamelCase)

In [0]:
SELECT emp_name, UPPER(emp_name) AS upper_name
FROM employee_payments

emp_name,upper_name
karthik,KARTHIK
veena,VEENA
ravi,RAVI
anil,ANIL
suresh,SURESH


In [0]:
SELECT emp_name, LOWER(emp_name) AS lower_name
FROM employee_payments

emp_name,lower_name
karthik,karthik
veena,veena
ravi,ravi
anil,anil
suresh,suresh


In [0]:
SELECT emp_name, INITCAP(emp_name) AS initcap_name
FROM employee_payments

emp_name,initcap_name
karthik,Karthik
veena,Veena
ravi,Ravi
anil,Anil
suresh,Suresh


#####Calculate total income = base_salary + bonus (NULL safe) + Round total income to nearest integer Extract joining year Use CASE to classify: Senior if experience > 7 years Mid if between 4 and 7 Junior otherwise


In [0]:
SELECT emp_name,
    ROUND(base_salary + IFNULL(bonus, 0)) AS total_income,
    YEAR(joining_date) AS joining_year,
    YEAR(CURDATE()) - YEAR(joining_date) AS experience,
    CASE 
        WHEN YEAR(CURDATE()) - YEAR(joining_date) > 7 THEN 'Senior'
        WHEN YEAR(CURDATE()) - YEAR(joining_date) BETWEEN 4 AND 7 THEN 'Mid'
        ELSE 'Junior'
    END AS emp_level
FROM employee_payments;

emp_name,total_income,joining_year,experience,emp_level
karthik,80001,2019,7,Mid
veena,69001,2021,5,Mid
ravi,91002,2016,10,Senior
anil,70000,2020,6,Mid
suresh,63001,2022,4,Mid


# QUESTION 2: Order Delivery Delay Analysis

In [0]:
CREATE TABLE orders_delivery (
order_id INT,
customer_name VARCHAR(50),
order_date DATE,
delivery_date DATE,
order_amount DECIMAL(10,2)
);
INSERT INTO orders_delivery VALUES
(101,'rajesh','2025-01-01','2025-01-05',12500.75),
(102,'meena','2025-01-10','2025-01-10',8400.40),
(103,'arun','2025-01-15','2025-01-20',15600.90),
(104,'pooja','2025-01-18',NULL,9200.10);

num_affected_rows,num_inserted_rows
4,4


##### For each order: Uppercase customer name Calculate delivery days using date difference Replace NULL delivery date with today Truncate order amount to 1 decimal
#####Use CASE: Same-day, Delayed (>3 days) ,Pending


In [0]:
SELECT order_id,
    UPPER(customer_name) AS customer_name,
    DATEDIFF(COALESCE(delivery_date, CURDATE()), order_date) AS delivery_days,
    COALESCE(delivery_date, CURDATE()) AS delivery_date,
    ROUND(order_amount, 1) AS truncated_amount,
    CASE 
        WHEN delivery_date IS NULL THEN 'Pending'
        WHEN DATEDIFF(delivery_date, order_date) = 0 THEN 'Same-day'
        WHEN DATEDIFF(delivery_date, order_date) > 3 THEN 'Delayed'
        ELSE 'On-time'
    END AS delivery_status
FROM orders_delivery;

order_id,customer_name,delivery_days,delivery_date,truncated_amount,delivery_status
101,RAJESH,4,2025-01-05,12500.8,Delayed
102,MEENA,0,2025-01-10,8400.4,Same-day
103,ARUN,5,2025-01-20,15600.9,Delayed
104,POOJA,449,2026-04-12,9200.1,Pending


# QUESTION 3: Customer Spending Pattern

In [0]:
CREATE TABLE customer_spending (
cust_id INT,
cust_name VARCHAR(50),
city VARCHAR(30),
purchase_amount DECIMAL(10,2),
purchase_date DATE
);
INSERT INTO customer_spending VALUES
(1,'amit','mumbai',12000.75,'2024-12-01'),
(2,'neha','delhi',8500.40,'2024-12-15'),
(3,'rohit','mumbai',15500.90,'2024-11-20'),
(4,'kavya','chennai',6000.10,'2024-10-05');

num_affected_rows,num_inserted_rows
4,4


##### Display: Customer name with first letter capitalized ,Month name of purchase ,Rounded purchase amount, Absolute value of purchase (defensive logic), CASE: ,High spender > 15000, Medium 8000–15000, Low otherwise


In [0]:
SELECT initcap(cust_name) AS cust_name,
    MONTHNAME(purchase_date) AS purchase_month,
    ROUND(purchase_amount) AS rounded_amount,
    ABS(purchase_amount) AS absolute_amount,
    CASE 
        WHEN purchase_amount > 15000 THEN 'High spender'
        WHEN purchase_amount BETWEEN 8000 AND 15000 THEN 'Medium'
        ELSE 'Low'
    END AS spending_category
FROM customer_spending;

cust_name,purchase_month,rounded_amount,absolute_amount,spending_category
Amit,Dec,12001,12000.75,Medium
Neha,Dec,8500,8500.40,Medium
Rohit,Nov,15501,15500.90,High spender
Kavya,Oct,6000,6000.10,Low


# QUESTION 4: Subscription Validity Check


In [0]:
CREATE TABLE subscriptions (
user_id INT,
user_email VARCHAR(100),
start_date DATE,
end_date DATE,
subscription_fee DECIMAL(10,2)
);
INSERT INTO subscriptions VALUES
(1,'karthik@gmail.com','2024-01-01','2025-01-01',12000.50),
(2,'veena@yahoo.com','2024-06-15','2024-12-15',8500.75),
(3,'ravi@hotmail.com','2023-03-01','2024-03-01',15000.90);

num_affected_rows,num_inserted_rows
3,3


###### For each user:, Extract email domain, Calculate subscription duration in months, Format fee with commas, Find remaining days from today, CASE:, Active, Expiring Soon (≤30 days), Expired


In [0]:
SELECT user_email,
    SUBSTRING(user_email, INSTR(user_email, '@') + 1) AS domain,
    TIMESTAMPDIFF(MONTH, start_date, end_date) AS duration_months,
    FORMAT_NUMBER(subscription_fee, 2) AS formatted_fee,
    DATEDIFF(CURDATE(), end_date) AS remaining_days,
    CASE 
        WHEN end_date < CURDATE() THEN 'Expired'
        WHEN DATEDIFF(end_date, CURDATE()) <= 30 THEN 'Expiring Soon'
        ELSE 'Active'
    END AS subscription_status
FROM subscriptions;

user_email,domain,duration_months,formatted_fee,remaining_days,subscription_status
karthik@gmail.com,gmail.com,12,"12,000.50",466,Expired
veena@yahoo.com,yahoo.com,6,"8,500.75",483,Expired
ravi@hotmail.com,hotmail.com,12,"15,000.90",772,Expired


# QUESTION 5: Loan EMI Risk Categorization

In [0]:
CREATE TABLE loan_details (
loan_id INT,
customer_name VARCHAR(50),
loan_amount DECIMAL(12,2),
interest_rate DECIMAL(5,2),
loan_start DATE
);
INSERT INTO loan_details VALUES
(201,'suresh',500000.75,8.5,'2022-01-10'),
(202,'mahesh',750000.40,9.2,'2021-05-20'),
(203,'anita',300000.90,7.8,'2023-07-01');

num_affected_rows,num_inserted_rows
3,3


##### compute the monthly EMI using a power function, calculate the number of years since the loan start date, round the EMI value, convert the customer name to uppercase, and classify each loan using a CASE statement as High Risk if the interest rate is greater than 9, Medium Risk if between 8 and 9, and Low Risk otherwise.

In [0]:
SELECT 
    loan_id,FLOOR(DATEDIFF(CURRENT_DATE(), loan_start)/365) AS loan_years,
    ROUND(
        (loan_amount * (interest_rate/100/12) * 
        POW(1 + (interest_rate/100/12), FLOOR(DATEDIFF(CURRENT_DATE(), loan_start)/365) * 12)) /
        (POW(1 + (interest_rate/100/12), FLOOR(DATEDIFF(CURRENT_DATE(), loan_start)/365) * 12) - 1)
    , 2) AS EMI,
    CASE 
        WHEN interest_rate > 9 THEN 'High Risk'
        WHEN interest_rate BETWEEN 8 AND 9 THEN 'Medium Risk'
        ELSE 'Low Risk'
    END AS risk_category
FROM loan_details;

loan_id,loan_years,EMI,risk_category
201,4,12324.17,Medium Risk
202,4,18735.1,High Risk
203,2,13540.88,Low Risk


# QUESTION 6: Employee Attendance Evaluation

In [0]:
CREATE TABLE attendance (
emp_id INT,
emp_name VARCHAR(50),
total_days INT,
present_days INT,
record_date DATE
);
INSERT INTO attendance VALUES
(1,'karthik',30,28,'2025-01-31'),
(2,'veena',30,22,'2025-01-31'),
(3,'ravi',30,18,'2025-01-31');

num_affected_rows,num_inserted_rows
3,3


##### Calculate attendance percentage (rounded), month name, difference between total and present days, lowercase employee name, and use CASE to classify as Excellent ≥90%, Average 75–89%, Poor otherwise.

In [0]:
SELECT 
    LOWER(emp_name) AS emp_name,
    MONTHNAME(record_date) AS month_name,
    ROUND((present_days * 100.0) / total_days, 2) AS attendance_percentage,
    (total_days - present_days) AS absent_days,
    CASE 
        WHEN (present_days * 100.0) / total_days >= 90 THEN 'Excellent'
        WHEN (present_days * 100.0) / total_days BETWEEN 75 AND 89 THEN 'Average'
        ELSE 'Poor'
    END AS performance
FROM attendance;

emp_name,month_name,attendance_percentage,absent_days,performance
karthik,Jan,93.33,2,Excellent
veena,Jan,73.33,8,Poor
ravi,Jan,60.00,12,Poor


# QUESTION 7: Product Discount Validation

In [0]:
CREATE TABLE product_sales (
product_id INT,
product_name VARCHAR(50),
mrp DECIMAL(10,2),
selling_price DECIMAL(10,2),
sale_date DATE
);
INSERT INTO product_sales VALUES
(1,'Laptop',75000.75,68000.50,'2025-01-10'),
(2,'Mobile',35000.40,33000.25,'2025-01-12'),
(3,'Tablet',25000.90,26000.75,'2025-01-15');

num_affected_rows,num_inserted_rows
3,3


#### Derive discount amount (absolute), discount percentage, day name of sale, proper case product name, and use CASE to classify as Valid Discount, Overpriced, or No Discount.

In [0]:
SELECT initcap(product_name)AS product_name,
    ABS(mrp - selling_price) AS discount_amount,
    ROUND((mrp - selling_price) * 100.0 / mrp, 2) AS discount_percentage,
    DAYNAME(sale_date) AS day_name,
    CASE 
        WHEN selling_price < mrp THEN 'Valid Discount'
        WHEN selling_price > mrp THEN 'Overpriced'
        ELSE 'No Discount'
    END AS discount_status
FROM product_sales;

product_name,discount_amount,discount_percentage,day_name,discount_status
Laptop,7000.25,9.33,Fri,Valid Discount
Mobile,2000.15,5.71,Sun,Valid Discount
Tablet,999.85,-4.00,Wed,Overpriced


# QUESTION 8: Insurance Policy Aging

In [0]:
CREATE TABLE insurance_policies (
policy_id INT,
holder_name VARCHAR(50),
premium_amount DECIMAL(10,2),
policy_start DATE,
policy_end DATE
);
INSERT INTO insurance_policies VALUES
(301,'arjun',12000.50,'2023-01-01','2026-01-01'),
(302,'megha',8500.75,'2022-06-15','2025-06-15'),
(303,'vinod',15000.90,'2021-03-01','2024-03-01');

num_affected_rows,num_inserted_rows
3,3


#### Show policy duration in years, remaining days, rounded premium, uppercase holder name, and use CASE to classify as Long Term, Mid Term, or Expired.

In [0]:
SELECT UPPER(holder_name) AS holder_name,
    FLOOR(DATEDIFF(policy_end, policy_start)/365) AS policy_years,
    DATEDIFF(policy_end, CURRENT_DATE()) AS remaining_days,
    ROUND(premium_amount) AS rounded_premium,
    CASE 
        WHEN policy_end < CURRENT_DATE() THEN 'Expired'
        WHEN FLOOR(DATEDIFF(policy_end, policy_start)/365) >= 3 THEN 'Long Term'
        ELSE 'Mid Term'
    END AS policy_status
FROM insurance_policies;

holder_name,policy_years,remaining_days,rounded_premium,policy_status
ARJUN,3,-101,12001,Expired
MEGHA,3,-301,8501,Expired
VINOD,3,-772,15001,Expired


# QUESTION 9: Salary Increment Simulation

In [0]:
CREATE TABLE salary_revision (
emp_id INT,
emp_name VARCHAR(50),
current_salary DECIMAL(10,2),
rating INT,
last_hike DATE
);
INSERT INTO salary_revision VALUES
(1,'karthik',75000.75,5,'2023-01-01'),
(2,'veena',65000.40,4,'2024-01-01'),
(3,'ravi',85000.90,3,'2022-01-01');

num_affected_rows,num_inserted_rows
3,3


#### Calculate years since last hike, increment using rating logic, new salary (rounded), lowercase employee name, and use CASE to classify as High Increment, Moderate, or No Increment.

In [0]:
SELECT LOWER(emp_name) AS emp_name,
    FLOOR(DATEDIFF(CURRENT_DATE(), last_hike)/365) AS years_since_hike,
    CASE 
        WHEN rating = 5 THEN current_salary * 0.20
        WHEN rating = 4 THEN current_salary * 0.10
        ELSE 0
    END AS increment_amount,
    ROUND(current_salary + 
        CASE 
            WHEN rating = 5 THEN current_salary * 0.20
            WHEN rating = 4 THEN current_salary * 0.10
            ELSE 0
        END
    ) AS new_salary,
    CASE 
        WHEN rating = 5 THEN 'High Increment'
        WHEN rating = 4 THEN 'Moderate'
        ELSE 'No Increment'
    END AS increment_status
FROM salary_revision;

emp_name,years_since_hike,increment_amount,new_salary,increment_status
karthik,3,15000.1500,90001,High Increment
veena,2,6500.0400,71500,Moderate
ravi,4,0.0000,85001,No Increment


# QUESTION 10: Customer Account Status Evaluation

In [0]:
CREATE TABLE bank_accounts (
account_id INT,
customer_name VARCHAR(50),
balance DECIMAL(12,2),
last_transaction DATE,
branch VARCHAR(30)
);
INSERT INTO bank_accounts VALUES
(501,'ramesh',125000.75,'2024-12-20','hyderabad'),
(502,'sita',8500.40,'2023-06-15','delhi'),
(503,'manoj',-2500.90,'2025-01-05','mumbai');

num_affected_rows,num_inserted_rows
3,3


##### Determine absolute balance, days since last transaction, proper case branch name, sign of balance, and use CASE to classify as Active, Dormant, or Overdrawn.

In [0]:
SELECT 
    customer_name,
    ABS(balance) AS absolute_balance,
    DATEDIFF(CURRENT_DATE(), last_transaction) AS days_since_transaction,
    initcap(branch) AS branch_name,
    SIGN(balance) AS balance_sign,
    CASE 
        WHEN balance < 0 THEN 'Overdrawn'
        WHEN DATEDIFF(CURRENT_DATE(), last_transaction) > 365 THEN 'Dormant'
        ELSE 'Active'
    END AS account_status

FROM bank_accounts;

customer_name,absolute_balance,days_since_transaction,branch_name,balance_sign,account_status
ramesh,125000.75,478,Hyderabad,1.0,Dormant
sita,8500.40,1032,Delhi,1.0,Dormant
manoj,2500.90,462,Mumbai,-1.0,Overdrawn


# Level -1


##  QUESTION 1 – Salary Risk Flagging Based on Tax Shock

In [0]:
CREATE TABLE salary_audit (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
tax_percent DECIMAL(5,2),
last_revision DATE
);
INSERT INTO salary_audit VALUES
(1,'karthik',75000.75,10.5,'2022-01-15'),
(2,'veena',65000.40,18.0,'2023-06-01'),
(3,'ravi',85000.90,25.0,'2020-11-20');

num_affected_rows,num_inserted_rows
3,3


##### for each employee normalize name to lowercase, calculate net salary after tax and round it, extract revision year, find months since revision, and use CASE: flag Tax Shock if tax > 20 AND months > 24, flag Review Needed if tax between 15–20, else Stable.

In [0]:
SELECT 
    emp_id,
    LOWER(emp_name) AS emp_name,
    ROUND(salary - (salary * tax_percent / 100), 0) AS net_salary,
    EXTRACT(YEAR FROM last_revision) AS revision_year,
    TIMESTAMPDIFF(MONTH, last_revision, CURDATE()) AS months_since_revision,
    CASE
        WHEN tax_percent > 20 AND TIMESTAMPDIFF(MONTH, last_revision, CURDATE()) > 24 
            THEN 'Tax Shock'
        WHEN tax_percent BETWEEN 15 AND 20 
            THEN 'Review Needed'
        ELSE 'Stable'
    END AS status
FROM salary_audit

emp_id,emp_name,net_salary,revision_year,months_since_revision,status
1,karthik,67126,2022,50,Stable
2,veena,53300,2023,34,Review Needed
3,ravi,63751,2020,64,Tax Shock


# QUESTION 2 – Bonus Abuse Detection

In [0]:
CREATE OR REPLACE TABLE bonus_monitor (
emp_code INT,
emp_name VARCHAR(50),
base_salary DECIMAL(10,2),
bonus DECIMAL(10,2),
bonus_date DATE
);
INSERT INTO bonus_monitor VALUES
(101,'Anil',70000.10,30000.00,'2025-01-10'),
(102,'Suresh',60000.55,3000.30,'2024-03-15'),
(103,'Ravi',85000.90,15000.75,'2023-12-01');

num_affected_rows,num_inserted_rows
3,3


##### For each record convert name to proper case, calculate bonus percentage of salary (rounded), extract day name of bonus, find absolute salary–bonus difference, and use CASE: flag Suspicious if bonus > 30% AND weekend, flag Normal if bonus <= 20%, else Audit.

In [0]:
SELECT emp_code,
    initcap(emp_name) emp_name,
    ROUND((bonus / base_salary) * 100, 0) AS bonus_percent,
    DAYNAME(bonus_date) AS day_name,
    ABS(base_salary - bonus) AS salary_bonus_diff,
    CASE
        WHEN (bonus / base_salary) * 100 > 30 
             AND DAYOFWEEK(bonus_date) IN (1,7) 
            THEN 'Suspicious'
        WHEN (bonus / base_salary) * 100 <= 20 
            THEN 'Normal'
        ELSE 'Audit'
    END AS status
FROM bonus_monitor;

emp_code,emp_name,bonus_percent,day_name,salary_bonus_diff,status
101,Anil,43,Fri,40000.10,Audit
102,Suresh,5,Fri,57000.25,Normal
103,Ravi,18,Fri,70000.15,Normal


# QUESTION 3 – Experience Parity Validation

In [0]:
CREATE TABLE employee_experience (
emp_id INT,
emp_name VARCHAR(50),
joining_date DATE,
declared_experience INT,
salary DECIMAL(10,2)
);
INSERT INTO employee_experience VALUES
(1,'Veena','2018-07-01',4,65000.40),
(2,'Ravi','2014-01-10',12,85000.90),
(3,'Anil','2020-09-01',3,70000.10);

num_affected_rows,num_inserted_rows
3,3


##### For each employee display the employee name in uppercase, calculate the actual experience from the joining date to the current date, find the difference between declared experience and actual experience, show the floored value of salary, and use a CASE statement to label the experience as Overstated if declared experience is greater than actual, Understated if declared experience is less than actual, or Matched if both are equal.

In [0]:
SELECT 
    UPPER(emp_name) AS emp_name_upper,
    FLOOR(DATEDIFF(CURDATE(), joining_date) / 365) AS actual_experience,
    declared_experience - FLOOR(DATEDIFF(CURDATE(), joining_date) / 365) AS experience_difference,
    FLOOR(salary) AS salary_floor,
    CASE 
        WHEN declared_experience > FLOOR(DATEDIFF(CURDATE(), joining_date) / 365) THEN 'Overstated'
        WHEN declared_experience < FLOOR(DATEDIFF(CURDATE(), joining_date) / 365) THEN 'Understated'
        ELSE 'Matched'
    END AS experience_status
FROM employee_experience

emp_name_upper,actual_experience,experience_difference,salary_floor,experience_status
VEENA,7,-3,65000,Understated
RAVI,12,0,85000,Matched
ANIL,5,-2,70000,Understated


# QUESTION 4 – Salary Digit Pattern Analysis


In [0]:
CREATE TABLE salary_digits (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
credit_date DATE
);
INSERT INTO salary_digits VALUES
(1,'Karthik',75000.75,'2025-01-01'),
(2,'Veena',65000.40,'2025-01-02'),
(3,'Suresh',60000.55,'2025-01-03');

num_affected_rows,num_inserted_rows
3,3


##### For each employee display the last two characters of the employee name, get the day of the month from the credit date, truncate the salary to an integer, apply MOD on salary, and use a CASE statement to show Pattern Match if salary MOD 10 equals the day of the month, otherwise No Match.

In [0]:
SELECT 
    RIGHT(emp_name, 2) AS last_two_chars,
    DAY(credit_date) AS day_of_month,
    FLOOR(salary) AS truncated_salary,
    MOD(FLOOR(salary), 10) AS salary_mod,
    CASE 
        WHEN MOD(FLOOR(salary), 10) = DAY(credit_date) THEN 'Pattern Match'
        ELSE 'No Match'
    END AS match_status
FROM salary_digits;

last_two_chars,day_of_month,truncated_salary,salary_mod,match_status
ik,1,75000,0,No Match
na,2,65000,0,No Match
sh,3,60000,0,No Match


# QUESTION 5 – Odd–Even Salary Compliance

In [0]:
CREATE TABLE payroll_control (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
payment_date DATE
);
INSERT INTO payroll_control VALUES
(1,'Ravi',85000.90,'2025-01-15'),
(2,'Anil',70000.10,'2025-01-16'),
(3,'Veena',65000.40,'2025-01-17');

num_affected_rows,num_inserted_rows
3,3


##### For each employee display the employee name in lowercase, extract the weekday from the payment date, round the salary, apply MOD on salary, and use a CASE statement to show Violation if even salary is paid on an odd weekday, otherwise Compliant.

In [0]:
SELECT 
    LOWER(emp_name) AS emp_name_lower,
    DATE_FORMAT(payment_date, 'E') AS weekday,
    ROUND(salary, 0) AS rounded_salary,
    MOD(ROUND(salary, 0), 2) AS salary_mod,
    CASE 
        WHEN MOD(ROUND(salary, 0), 2) = 0 
             AND DAYOFWEEK(payment_date) % 2 = 1 THEN 'Violation'
        ELSE 'Compliant'
    END AS status
FROM payroll_control;

emp_name_lower,weekday,rounded_salary,salary_mod,status
ravi,Wed,85001,1,Compliant
anil,Thu,70000,0,Violation
veena,Fri,65000,0,Compliant


# QUESTION 6 – Salary Inflation Drift

In [0]:
CREATE TABLE inflation_watch (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
last_hike DATE
);
INSERT INTO inflation_watch VALUES
(1,'Karthik',75000.75,'2019-01-01'),
(2,'Veena',65000.40,'2022-01-01'),
(3,'Ravi',85000.90,'2017-01-01');

num_affected_rows,num_inserted_rows
3,3


##### For each employee display the employee name in proper case, calculate the number of years since the last hike, apply POWER on the years, round the salary impact, and use a CASE statement to classify as High Inflation Risk if years are greater than 5, Moderate if in between, or Low otherwise.

In [0]:
SELECT 
    INITCAP(emp_name) AS proper_name,
    FLOOR(DATEDIFF(CURDATE(), last_hike) / 365) AS years_since_hike,
    POWER(FLOOR(DATEDIFF(CURDATE(), last_hike) / 365), 2) AS power_years,
    ROUND(salary * POWER(FLOOR(DATEDIFF(CURDATE(), last_hike) / 365), 2), 0) AS salary_impact,
    CASE 
        WHEN FLOOR(DATEDIFF(CURDATE(), last_hike) / 365) > 5 THEN 'High Inflation Risk'
        WHEN FLOOR(DATEDIFF(CURDATE(), last_hike) / 365) BETWEEN 3 AND 5 THEN 'Moderate'
        ELSE 'Low'
    END AS risk_level
FROM inflation_watch;

proper_name,years_since_hike,power_years,salary_impact,risk_level
Karthik,7,49.0,3675037.0,High Inflation Risk
Veena,4,16.0,1040006.0,Moderate
Ravi,9,81.0,6885073.0,High Inflation Risk


# QUESTION 7 – Salary Sign Integrity Check

In [0]:
CREATE TABLE salary_integrity (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
record_date DATE
);
INSERT INTO salary_integrity VALUES
(1,'Anil',-70000.10,'2025-01-10'),
(2,'Veena',65000.40,'2025-01-10'),
(3,'Ravi',0.00,'2025-01-10');

num_affected_rows,num_inserted_rows
3,3


#### For each employee display the employee name in uppercase, extract the year from the record date, apply SIGN on salary, find the absolute salary, and use a CASE statement to classify as Negative Error if salary is negative, Zero Salary if salary is zero, or Valid otherwise.

In [0]:
SELECT 
    UPPER(emp_name) AS emp_name_upper,
    YEAR(record_date) AS year,
    SIGN(salary) AS salary_sign,
    ABS(salary) AS absolute_salary,
    CASE 
        WHEN salary < 0 THEN 'Negative Error'
        WHEN salary = 0 THEN 'Zero Salary'
        ELSE 'Valid'
    END AS status
FROM salary_integrity

emp_name_upper,year,salary_sign,absolute_salary,status
ANIL,2025,-1.0,70000.10,Negative Error
VEENA,2025,1.0,65000.40,Valid
RAVI,2025,0.0,0.00,Zero Salary


# QUESTION 8 – Name Length vs Salary Correlation


In [0]:
CREATE TABLE name_salary (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
join_date DATE
);

INSERT INTO name_salary VALUES
(1,'Karthik',75000.75,'2019-03-15'),
(2,'Veena',65000.40,'2021-06-20'),
(3,'Ravi',85000.90,'2016-01-10');

num_affected_rows,num_inserted_rows
3,3


##### For each employee calculate the length of the employee name, calculate the years of service from the join date, round the salary, compare the name length with years of service, and use a CASE statement to classify as Name Bias if name length is greater than years, otherwise Neutral.

In [0]:
SELECT 
    LENGTH(emp_name) AS name_length,
    FLOOR(DATEDIFF(CURDATE(), join_date) / 365) AS years_of_service,
    ROUND(salary, 0) AS rounded_salary,
    CASE 
        WHEN LENGTH(emp_name) > FLOOR(DATEDIFF(CURDATE(), join_date) / 365) THEN 'Name Bias'
        ELSE 'Neutral'
    END AS status
FROM name_salary;

name_length,years_of_service,rounded_salary,status
7,7,75001,Neutral
5,4,65000,Name Bias
4,10,85001,Neutral


# QUESTION 9 – Salary Spike Detection by Month

In [0]:
CREATE TABLE salary_monthly (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
paid_date DATE
);
INSERT INTO salary_monthly VALUES
(1,'Karthik',75000.75,'2025-01-31'),
(2,'Veena',65000.40,'2025-02-28'),
(3,'Ravi',85000.90,'2025-03-31');

num_affected_rows,num_inserted_rows
3,3


#### For each record extract the month name from the paid date, apply CEIL on salary, check whether the paid date is the last day of the month, and use a CASE statement to classify as End Month Spike if it is the last day, otherwise Regular.

In [0]:
SELECT 
    DATE_FORMAT(paid_date, 'MMMM') AS month_name,
    CEIL(salary) AS ceil_salary,
    LAST_DAY(paid_date) AS last_day_of_month,
    CASE 
        WHEN paid_date = LAST_DAY(paid_date) THEN 'End Month Spike'
        ELSE 'Regular'
    END AS status
FROM salary_monthly;

month_name,ceil_salary,last_day_of_month,status
January,75001,2025-01-31,End Month Spike
February,65001,2025-02-28,End Month Spike
March,85001,2025-03-31,End Month Spike


# QUESTION 10 – Salary Digit Sum Audit

In [0]:
CREATE TABLE digit_audit (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
audit_date DATE
);

INSERT INTO digit_audit VALUES
(1,'Anil',70000.10,'2025-01-01'),
(2,'Veena',65000.40,'2025-01-02');

num_affected_rows,num_inserted_rows
2,2


##### For each employee extract the first character of the employee name, truncate the salary to an integer, logically sum the digits of the salary, extract the day from the audit date, and use a CASE statement to classify as Digit Alert if the sum of digits matches a condition with the day, otherwise Normal.

In [0]:
SELECT 
    SUBSTRING(emp_name, 1, 1) AS first_char,
    CAST(salary AS INT) AS truncated_salary,
    (
        (CAST(salary AS INT) % 10) +
        ((CAST(salary AS INT) / 10) % 10) +
        ((CAST(salary AS INT) / 100) % 10) +
        ((CAST(salary AS INT) / 1000) % 10) +
        ((CAST(salary AS INT) / 10000) % 10)
    ) AS digit_sum,
    DAY(audit_date) AS day_value,
    CASE 
        WHEN (
            (CAST(salary AS INT) % 10) +
            ((CAST(salary AS INT) / 10) % 10) +
            ((CAST(salary AS INT) / 100) % 10) +
            ((CAST(salary AS INT) / 1000) % 10) +
            ((CAST(salary AS INT) / 10000) % 10)
        ) = DAY(audit_date) THEN 'Digit Alert'
        ELSE 'Normal'
    END AS status
FROM digit_audit;

first_char,truncated_salary,digit_sum,day_value,status
A,70000,7.0,1,Normal
V,65000,11.5,2,Normal


##  QUESTION 11 – Weekend Salary Credit Fraud Detection

In [0]:
CREATE TABLE salary_credit_audit (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
credit_date DATE,
bank_code VARCHAR(10)
);
INSERT INTO salary_credit_audit VALUES
(1,'Karthik',75000.75,'2025-01-04','HDFC01'),
(2,'Veena',65000.40,'2025-01-06','ICIC02'),
(3,'Ravi',85000.90,'2025-01-05','SBIN03'),
(4,'Anil',70000.10,'2025-01-07','AXIS04'),
(5,'Suresh',60000.55,'2025-01-11','HDFC01');

num_affected_rows,num_inserted_rows
5,5


##### For each record extract the bank prefix from the bank code, identify the weekday name of the credit date, round the salary, apply MOD on salary, and use a CASE statement to classify as Weekend Fraud if credited on Saturday or Sunday and salary MOD 5 equals 0, Bank Review if the bank is HDFC, otherwise Normal.

In [0]:
SELECT 
    SUBSTRING(bank_code, 1, 4) AS bank_prefix,
    DATE_FORMAT(credit_date, 'EEEE') AS weekday_name,
    ROUND(salary, 0) AS rounded_salary,
    MOD(ROUND(salary, 0), 5) AS salary_mod,  
    CASE 
        WHEN DAYOFWEEK(credit_date) IN (1,7) 
             AND MOD(ROUND(salary, 0), 5) = 0 
        THEN 'Weekend Fraud'   
        WHEN SUBSTRING(bank_code, 1, 4) = 'HDFC' 
        THEN 'Bank Review'     
        ELSE 'Normal'
    END AS status
FROM salary_credit_audit;

bank_prefix,weekday_name,rounded_salary,salary_mod,status
HDFC,Saturday,75001,1,Bank Review
ICIC,Monday,65000,0,Normal
SBIN,Sunday,85001,1,Normal
AXIS,Tuesday,70000,0,Normal
HDFC,Saturday,60001,1,Bank Review


# QUESTION 12 – Salary Credit Time Drift Analysis

In [0]:
CREATE TABLE salary_time_drift (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
credit_ts TIMESTAMP
);

INSERT INTO salary_time_drift VALUES
(1,'Karthik',75000.75,'2025-01-10 23:45:00'),
(2,'Veena',65000.40,'2025-01-10 09:15:00'),
(3,'Ravi',85000.90,'2025-01-11 00:10:00'),
(4,'Anil',70000.10,'2025-01-09 18:30:00'),
(5,'Suresh',60000.55,'2025-01-10 02:50:00');

num_affected_rows,num_inserted_rows
5,5


##### For each employee extract the hour from the credit timestamp, convert the employee name to lowercase, floor the salary, calculate the difference between salary and hour, and use a CASE statement to classify as Midnight Drift if hour is between 0 and 3, After Hours if outside business hours, or Business Hours otherwise.

In [0]:
SELECT 
    HOUR(credit_ts) AS hour_value,
    LOWER(emp_name) AS emp_name_lower,
    FLOOR(salary) AS floored_salary,
    FLOOR(salary) - HOUR(credit_ts) AS salary_hour_diff,  
    CASE 
        WHEN HOUR(credit_ts) BETWEEN 0 AND 3 THEN 'Midnight Drift'
        WHEN HOUR(credit_ts) < 9 OR HOUR(credit_ts) > 18 THEN 'After Hours'
        ELSE 'Business Hours'
    END AS status
FROM salary_time_drift;

hour_value,emp_name_lower,floored_salary,salary_hour_diff,status
23,karthik,75000,74977,After Hours
9,veena,65000,64991,Business Hours
0,ravi,85000,85000,Midnight Drift
18,anil,70000,69982,Business Hours
2,suresh,60000,59998,Midnight Drift


# QUESTION 13 – Salary Decimal Precision Audit

In [0]:
CREATE TABLE salary_precision (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,4),
record_date DATE
);
INSERT INTO salary_precision VALUES
(1,'Karthik',75000.7567,'2025-01-01'),
(2,'Veena',65000.4044,'2025-01-02'),
(3,'Ravi',85000.9099,'2025-01-03'),
(4,'Anil',70000.1001,'2025-01-04'),
(5,'Suresh',60000.5555,'2025-01-05');

num_affected_rows,num_inserted_rows
5,5


###### For each record truncate the salary to 2 decimal places, calculate the difference between rounded and truncated salary values, extract the day name from the record date, get the length of the employee name, and use a CASE statement to classify as Precision Loss if the difference is greater than 0.01, otherwise Safe.

In [0]:
SELECT 
    FLOOR(salary * 100) / 100 AS truncated_salary,
    ROUND(salary, 2) - (FLOOR(salary * 100) / 100) AS diff_value,
    DATE_FORMAT(record_date, 'EEEE') AS day_name,
    LENGTH(emp_name) AS name_length,   
    CASE 
        WHEN (ROUND(salary, 2) - (FLOOR(salary * 100) / 100)) > 0.01 
        THEN 'Precision Loss'
        ELSE 'Safe'
    END AS status
FROM salary_precision;

truncated_salary,diff_value,day_name,name_length,status
75000.750000,0.010000,Wednesday,7,Safe
65000.400000,0.000000,Thursday,5,Safe
85000.900000,0.010000,Friday,4,Safe
70000.100000,0.000000,Saturday,4,Safe
60000.550000,0.010000,Sunday,6,Safe


# QUESTION 14 – Salary Growth Power Index


In [0]:
CREATE TABLE salary_growth (
emp_id INT,
emp_name VARCHAR(50),
base_salary DECIMAL(10,2),
growth_rate DECIMAL(5,2),
last_hike DATE
);
INSERT INTO salary_growth VALUES
(1,'Karthik',75000.75,1.08,'2019-01-01'),
(2,'Veena',65000.40,1.05,'2021-01-01'),
(3,'Ravi',85000.90,1.12,'2017-01-01'),
(4,'Anil',70000.10,1.03,'2022-01-01'),
(5,'Suresh',60000.55,1.06,'2020-01-01');

num_affected_rows,num_inserted_rows
5,5


###### For each employee calculate the number of years since the last hike, apply POWER using growth rate and years, round the projected salary, convert employee name to uppercase, and use a CASE statement to classify as Explosive Growth if projected salary is greater than 150000, Controlled if moderate, or Stagnant otherwise.

In [0]:
SELECT 
    UPPER(emp_name) AS emp_name_upper,
    FLOOR(DATEDIFF(CURDATE(), last_hike) / 365) AS years_since_hike,
    POWER(growth_rate, FLOOR(DATEDIFF(CURDATE(), last_hike) / 365)) AS growth_factor,
    ROUND(base_salary * POWER(growth_rate, FLOOR(DATEDIFF(CURDATE(), last_hike) / 365)),0) AS projected_salary,
    CASE 
        WHEN ROUND(base_salary * POWER(growth_rate, FLOOR(DATEDIFF(CURDATE(), last_hike) / 365)),0) > 150000 
        THEN 'Explosive Growth'
        WHEN ROUND(base_salary * POWER(growth_rate, FLOOR(DATEDIFF(CURDATE(), last_hike) / 365)),0) BETWEEN 100000 AND 150000
        THEN 'Controlled'
        ELSE 'Stagnant'
    END AS status
FROM salary_growth;

emp_name_upper,years_since_hike,growth_factor,projected_salary,status
KARTHIK,7,1.7138242687795209,128538.0,Controlled
VEENA,5,1.2762815625000004,82959.0,Stagnant
RAVI,9,2.773078757450189,235714.0,Explosive Growth
ANIL,4,1.1255088100000001,78786.0,Stagnant
SURESH,6,1.4185191122560004,85112.0,Stagnant


# QUESTION 15 – Salary Symmetry Check

In [0]:
CREATE TABLE salary_symmetry (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
processed_date DATE
);
INSERT INTO salary_symmetry VALUES
(1,'Karthik',75557.75,'2025-01-15'),
(2,'Veena',64446.40,'2025-01-16'),
(3,'Ravi',85858.90,'2025-01-17'),
(4,'Anil',70007.10,'2025-01-18'),
(5,'Suresh',60000.55,'2025-01-19');

num_affected_rows,num_inserted_rows
5,5


###### For each record remove decimals from the salary, reverse the digits of the salary, extract the weekday from the processed date, convert employee name to proper case, and use a CASE statement to classify as Symmetric Pay if the reversed salary equals the original salary, otherwise Asymmetric.

In [0]:
SELECT 
    FLOOR(salary) AS salary_no_decimal,
    REVERSE(CAST(FLOOR(salary) AS STRING)) AS reversed_salary,
    DATE_FORMAT(processed_date, 'EEEE') AS weekday,
    INITCAP(emp_name) AS proper_name, 
    CASE 
        WHEN CAST(FLOOR(salary) AS STRING) = REVERSE(CAST(FLOOR(salary) AS STRING))
        THEN 'Symmetric Pay'
        ELSE 'Asymmetric'
    END AS status
FROM salary_symmetry;

salary_no_decimal,reversed_salary,weekday,proper_name,status
75557,75557,Wednesday,Karthik,Symmetric Pay
64446,64446,Thursday,Veena,Symmetric Pay
85858,85858,Friday,Ravi,Symmetric Pay
70007,70007,Saturday,Anil,Symmetric Pay
60000,00006,Sunday,Suresh,Asymmetric


# QUESTION 16 – Leap Year Salary Adjustment Audit

In [0]:
CREATE TABLE leap_salary (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
credit_date DATE
);
INSERT INTO leap_salary VALUES
(1,'Karthik',75000.75,'2024-02-29'),
(2,'Veena',65000.40,'2025-02-28'),
(3,'Ravi',85000.90,'2020-02-29'),
(4,'Anil',70000.10,'2023-02-28'),
(5,'Suresh',60000.55,'2024-02-28');

num_affected_rows,num_inserted_rows
5,5


###### For each employee extract the year from the credit date, check leap year logic, apply CEIL on salary, calculate the day of the year, and use a CASE statement to classify as Leap Credit if it is a leap year date, otherwise Non-Leap Credit.

In [0]:
SELECT 
    YEAR(credit_date) AS year_value,
    CASE 
        WHEN (YEAR(credit_date) % 4 = 0 AND YEAR(credit_date) % 100 != 0)
             OR (YEAR(credit_date) % 400 = 0)
        THEN 'Leap Year'
        ELSE 'Non-Leap Year'
    END AS leap_flag,
    CEIL(salary) AS ceil_salary,
    DAYOFYEAR(credit_date) AS day_of_year,   
    CASE 
        WHEN (YEAR(credit_date) % 4 = 0 AND YEAR(credit_date) % 100 != 0)
             OR (YEAR(credit_date) % 400 = 0)
        THEN 'Leap Credit'
        ELSE 'Non-Leap Credit'
    END AS status
FROM leap_salary;

year_value,leap_flag,ceil_salary,day_of_year,status
2024,Leap Year,75001,60,Leap Credit
2025,Non-Leap Year,65001,59,Non-Leap Credit
2020,Leap Year,85001,60,Leap Credit
2023,Non-Leap Year,70001,59,Non-Leap Credit
2024,Leap Year,60001,59,Leap Credit


# QUESTION 17 – Fiscal Year Boundary Salary Check

In [0]:
CREATE TABLE fiscal_salary (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
credit_date DATE
);
INSERT INTO fiscal_salary VALUES
(1,'Karthik',75000.75,'2025-03-31'),
(2,'Veena',65000.40,'2025-04-01'),
(3,'Ravi',85000.90,'2024-03-30'),
(4,'Anil',70000.10,'2024-04-02'),
(5,'Suresh',60000.55,'2025-03-29');

num_affected_rows,num_inserted_rows
5,5


##### For each record determine the fiscal year from the credit date, extract the month, format the salary, convert the employee name to lowercase, and use a CASE statement to classify as Year End Credit if it falls at the end of the fiscal year, Year Start Credit if at the beginning, otherwise Mid Year.

In [0]:
SELECT 
    -- Fiscal year (April to March)
    CASE 
        WHEN MONTH(credit_date) >= 4 
        THEN CONCAT(YEAR(credit_date), '-', YEAR(credit_date) + 1)
        ELSE CONCAT(YEAR(credit_date) - 1, '-', YEAR(credit_date))
    END AS fiscal_year,
    MONTH(credit_date) AS month_value,
    FORMAT_NUMBER(salary, 2) AS formatted_salary,
    LOWER(emp_name) AS emp_name_lower,
    CASE 
        WHEN MONTH(credit_date) = 3 THEN 'Year End Credit'
        WHEN MONTH(credit_date) = 4 THEN 'Year Start Credit'
        ELSE 'Mid Year'
    END AS status
FROM fiscal_salary;

fiscal_year,month_value,formatted_salary,emp_name_lower,status
2024-2025,3,"75,000.75",karthik,Year End Credit
2025-2026,4,"65,000.40",veena,Year Start Credit
2023-2024,3,"85,000.90",ravi,Year End Credit
2024-2025,4,"70,000.10",anil,Year Start Credit
2024-2025,3,"60,000.55",suresh,Year End Credit


# QUESTION 18 – Salary Random Sampling for Audit

In [0]:
CREATE TABLE salary_sampling (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
record_date DATE
);
INSERT INTO salary_sampling VALUES
(1,'Karthik',75000.75,'2025-01-01'),
(2,'Veena',65000.40,'2025-01-02'),
(3,'Ravi',85000.90,'2025-01-03'),
(4,'Anil',70000.10,'2025-01-04'),
(5,'Suresh',60000.55,'2025-01-05'),
(6,'Amit',72000.60,'2025-01-06'),
(7,'Neha',68000.80,'2025-01-07');

num_affected_rows,num_inserted_rows
7,7


##### For each record generate a random value, round the salary, extract the day name from the record date, extract the first character of the employee name, and use a CASE statement to classify as Sampled if RAND() is greater than 0.7, otherwise Skipped.

In [0]:
SELECT 
    RAND() AS random_value,
    ROUND(salary, 0) AS rounded_salary,
    DATE_FORMAT(record_date, 'EEEE') AS day_name,
    SUBSTRING(emp_name, 1, 1) AS first_char,
    CASE 
        WHEN RAND() > 0.7 THEN 'Sampled'
        ELSE 'Skipped'
    END AS status
FROM salary_sampling;

random_value,rounded_salary,day_name,first_char,status
0.49309487987702394,75001,Wednesday,K,Skipped
0.5476548901659947,65000,Thursday,V,Skipped
0.6964283888203041,85001,Friday,R,Sampled
0.05782196920857019,70000,Saturday,A,Skipped
0.796173010194778,60001,Sunday,S,Skipped
0.3371275848594041,72001,Monday,A,Sampled
0.7427421602397088,68001,Tuesday,N,Skipped


# QUESTION 19 – Salary ASCII Integrity Check

In [0]:
CREATE TABLE salary_ascii (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
join_date DATE
);
INSERT INTO salary_ascii VALUES
(1,'Karthik',75000.75,'2019-03-15'),
(2,'Veena',65000.40,'2021-06-20'),
(3,'Ravi',85000.90,'2016-01-10'),
(4,'Anil',70000.10,'2020-09-01'),
(5,'Suresh',60000.55,'2022-11-25');

num_affected_rows,num_inserted_rows
5,5


###### For each employee extract the ASCII value of the first character of the employee name, calculate the years since joining from the join date, floor the salary, compare the ASCII value with years of service, and use a CASE statement to classify as Name Dominates if ASCII value is greater than years, otherwise Experience Dominates.

In [0]:
SELECT 
    ASCII(SUBSTRING(emp_name, 1, 1)) AS ascii_value,
    FLOOR(DATEDIFF(CURDATE(), join_date) / 365) AS years_of_service,
    FLOOR(salary) AS floored_salary, 
    CASE 
        WHEN ASCII(SUBSTRING(emp_name, 1, 1)) > FLOOR(DATEDIFF(CURDATE(), join_date) / 365)
        THEN 'Name Dominates'
        ELSE 'Experience Dominates'
    END AS status
FROM salary_ascii;

ascii_value,years_of_service,floored_salary,status
75,7,75000,Name Dominates
86,4,65000,Name Dominates
82,10,85000,Name Dominates
65,5,70000,Name Dominates
83,3,60000,Name Dominates


# QUESTION 20 – Salary vs Calendar Symmetry Logic

In [0]:
CREATE TABLE salary_calendar (
emp_id INT,
emp_name VARCHAR(50),
salary DECIMAL(10,2),
credit_date DATE
);
INSERT INTO salary_calendar VALUES
(1,'Karthik',75000.75,'2025-01-15'),
(2,'Veena',65000.40,'2025-02-14'),
(3,'Ravi',85000.90,'2025-03-31'),
(4,'Anil',70000.10,'2025-04-04'),
(5,'Suresh',60000.55,'2025-05-05');

num_affected_rows,num_inserted_rows
5,5


###### For each record extract the day and month from the credit date, extract the last two digits of the salary, convert employee name to uppercase, calculate the absolute difference between day and month, and use a CASE statement to classify as Calendar Match if day equals month or salary last two digits match, otherwise Calendar Drift.

In [0]:
SELECT 
    DAY(credit_date) AS day_value,
    MONTH(credit_date) AS month_value,
    MOD(CAST(FLOOR(salary) AS INT), 100) AS last_two_digits, 
    UPPER(emp_name) AS emp_name_upper, 
    ABS(DAY(credit_date) - MONTH(credit_date)) AS day_month_diff, 
    CASE 
        WHEN DAY(credit_date) = MONTH(credit_date)
             OR MOD(CAST(FLOOR(salary) AS INT), 100) = DAY(credit_date)
        THEN 'Calendar Match'
        ELSE 'Calendar Drift'
    END AS status
FROM salary_calendar;

day_value,month_value,last_two_digits,emp_name_upper,day_month_diff,status
15,1,0,KARTHIK,14,Calendar Drift
14,2,0,VEENA,12,Calendar Drift
31,3,0,RAVI,28,Calendar Drift
4,4,0,ANIL,0,Calendar Match
5,5,0,SURESH,0,Calendar Match


# LEVEL -2

## QUESTION 1 – Employee Login Discipline & Performance Classification

In [0]:
CREATE TABLE employee_login (
emp_id INT,
emp_name VARCHAR(50),
login_time TIMESTAMP,
logout_time TIMESTAMP
);
INSERT INTO employee_login VALUES
(1,'Karthik','2025-01-15 09:05:00','2025-01-15 18:10:00'),
(2,'Veena','2025-01-14 10:30:00','2025-01-14 16:00:00'),
(3,'Ravi','2025-01-13 09:00:00','2025-01-13 20:00:00'),
(4,'Anil','2025-01-12 11:00:00','2025-01-12 14:00:00'),
(5,'Suresh','2025-01-11 09:15:00','2025-01-11 17:00:00');

num_affected_rows,num_inserted_rows
5,5


###### For each employee convert the employee name to proper case, identify whether the login date is a weekday or weekend, calculate total working hours from logout and login time, round working hours to 2 decimal places, and use a CASE statement to classify as Good Performer if weekday and working hours are greater than or equal to 8, Bad Performer if weekday and working hours are less than 6, otherwise Weekend Login

In [0]:
SELECT 
    INITCAP(emp_name) AS proper_name,   
    CASE 
        WHEN DAYOFWEEK(login_time) IN (1,7) THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,  
    ROUND((UNIX_TIMESTAMP(logout_time) - UNIX_TIMESTAMP(login_time)) / 3600, 2) AS working_hours,  
    CASE 
        WHEN DAYOFWEEK(login_time) NOT IN (1,7) 
             AND ROUND((UNIX_TIMESTAMP(logout_time) - UNIX_TIMESTAMP(login_time)) / 3600, 2) >= 8 
        THEN 'Good Performer' 
        WHEN DAYOFWEEK(login_time) NOT IN (1,7) 
             AND ROUND((UNIX_TIMESTAMP(logout_time) - UNIX_TIMESTAMP(login_time)) / 3600, 2) < 6 
        THEN 'Bad Performer'      
        ELSE 'Weekend Login'
    END AS status
FROM employee_login;

proper_name,day_type,working_hours,status
Karthik,Weekday,9.08,Good Performer
Veena,Weekday,5.5,Bad Performer
Ravi,Weekday,11.0,Good Performer
Anil,Weekend,3.0,Weekend Login
Suresh,Weekend,7.75,Weekend Login


##  QUESTION 2 – Past 7 Days Attendance & Productivity Check

In [0]:
CREATE TABLE attendance_log (
emp_id INT,
emp_name VARCHAR(50),
login_date DATE,
login_time TIMESTAMP,
logout_time TIMESTAMP
);
INSERT INTO attendance_log VALUES
(1,'Karthik','2025-01-14','09:00:00','18:00:00'),
(2,'Karthik','2025-01-13','09:15:00','17:30:00'),
(3,'Veena','2025-01-12','10:00:00','15:00:00'),
(4,'Ravi','2025-01-10','09:00:00','19:00:00'),
(5,'Anil','2025-01-08','11:00:00','14:00:00');

num_affected_rows,num_inserted_rows
5,5


###### For each record convert the employee name to uppercase, check if the login date falls within the last 7 days from today, identify whether it is a weekday or weekend, calculate working hours using time difference, and use a CASE statement to classify as Active & Productive if within last 7 days and working hours are greater than or equal to 8, Active but Low Hours if within last 7 days and working hours are less than 8, otherwise Absent from Last 7 Days.

In [0]:
SELECT 
    UPPER(emp_name) AS emp_name_upper, 
    CASE 
        WHEN login_date >= DATE_SUB(CURDATE(), 7) THEN 'Last 7 Days'
        ELSE 'Older'
    END AS recent_flag,    
    CASE 
        WHEN DAYOFWEEK(login_date) IN (1,7) THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,   
    ROUND(
        (UNIX_TIMESTAMP(TO_TIMESTAMP(logout_time)) - 
         UNIX_TIMESTAMP(TO_TIMESTAMP(login_time))) / 3600,
        2
    ) AS working_hours,   
    CASE 
        WHEN login_date >= DATE_SUB(CURDATE(), 7)
             AND ROUND(
                 (UNIX_TIMESTAMP(TO_TIMESTAMP(logout_time)) - 
                  UNIX_TIMESTAMP(TO_TIMESTAMP(login_time))) / 3600,
                 2
             ) >= 8
        THEN 'Active & Productive'      
        WHEN login_date >= DATE_SUB(CURDATE(), 7)
             AND ROUND(
                 (UNIX_TIMESTAMP(TO_TIMESTAMP(logout_time)) - 
                  UNIX_TIMESTAMP(TO_TIMESTAMP(login_time))) / 3600,
                 2
             ) < 8
        THEN 'Active but Low Hours'        
        ELSE 'Absent from Last 7 Days'
    END AS status
FROM attendance_log;

emp_name_upper,recent_flag,day_type,working_hours,status
KARTHIK,Older,Weekday,9.0,Absent from Last 7 Days
KARTHIK,Older,Weekday,8.25,Absent from Last 7 Days
VEENA,Older,Weekend,5.0,Absent from Last 7 Days
RAVI,Older,Weekday,10.0,Absent from Last 7 Days
ANIL,Older,Weekday,3.0,Absent from Last 7 Days


# QUESTION 3 – Weekend Work Abuse Detection

In [0]:
CREATE or replace TABLE weekend_monitor (
emp_id INT,
emp_name VARCHAR(50),
work_date DATE,
login_time TIMESTAMP,
logout_time TIMESTAMP
);
INSERT INTO weekend_monitor VALUES
(1,'Ravi','2025-01-11','2025-01-11 09:00:00','2025-01-11 21:00:00'),
(2,'Veena','2025-01-12','2025-01-12 10:00:00','2025-01-12 13:00:00'),
(3,'Karthik','2025-01-10','2025-01-10 09:00:00','2025-01-10 18:00:00'),
(4,'Anil','2025-01-09','2025-01-09 11:00:00','2025-01-09 14:00:00');

num_affected_rows,num_inserted_rows
4,4


###### For each employee extract the day name from the work date, convert the employee name to lowercase, calculate working hours from login and logout timestamps, apply CEIL on working hours, and use a CASE statement to classify as Weekend Overtime if it is Saturday or Sunday and hours are greater than or equal to 8, Suspicious Login if it is weekend and hours are less than 4, otherwise Normal Working Day.

In [0]:
SELECT 
    DATE_FORMAT(work_date, 'EEEE') AS day_name,
    LOWER(emp_name) AS emp_name_lower,
    (UNIX_TIMESTAMP(logout_time) - UNIX_TIMESTAMP(login_time)) / 3600 AS working_hours,
    CEIL((UNIX_TIMESTAMP(logout_time) - UNIX_TIMESTAMP(login_time)) / 3600) AS ceil_hours,
    CASE 
        WHEN DAYOFWEEK(work_date) IN (1,7)
             AND ((UNIX_TIMESTAMP(logout_time) - UNIX_TIMESTAMP(login_time)) / 3600) >= 8
        THEN 'Weekend Overtime'  
        WHEN DAYOFWEEK(work_date) IN (1,7)
             AND ((UNIX_TIMESTAMP(logout_time) - UNIX_TIMESTAMP(login_time)) / 3600) < 4
        THEN 'Suspicious Login'  
        ELSE 'Normal Working Day'
    END AS status
FROM weekend_monitor;

day_name,emp_name_lower,working_hours,ceil_hours,status
Saturday,ravi,12.0,12,Weekend Overtime
Sunday,veena,3.0,3,Suspicious Login
Friday,karthik,9.0,9,Normal Working Day
Thursday,anil,3.0,3,Normal Working Day


## QUESTION 4 – Login Time Deviation & Discipline Score

In [0]:
CREATE TABLE login_discipline (
emp_id INT,
emp_name VARCHAR(50),
login_datetime TIMESTAMP,
logout_datetime TIMESTAMP
);
INSERT INTO login_discipline VALUES
(1,'Karthik','2025-01-15 08:55:00','2025-01-15 18:10:00'),
(2,'Veena','2025-01-15 10:45:00','2025-01-15 16:00:00'),
(3,'Ravi','2025-01-15 09:00:00','2025-01-15 20:30:00'),
(4,'Anil','2025-01-15 11:30:00','2025-01-15 14:00:00');

num_affected_rows,num_inserted_rows
4,4


###### For each employee extract the login hour from the login timestamp, calculate total working hours from login and logout time, truncate working hours to 1 decimal place, get the weekday name, and use a CASE statement to classify as Disciplined if weekday and login before 9 and working hours are greater than or equal to 8, Late Comer if weekday and login after 10, otherwise Poor Discipline.

In [0]:
SELECT 
    HOUR(login_datetime) AS login_hour, 
    (UNIX_TIMESTAMP(logout_datetime) - UNIX_TIMESTAMP(login_datetime)) / 3600 AS working_hours,
    FLOOR(((UNIX_TIMESTAMP(logout_datetime) - UNIX_TIMESTAMP(login_datetime)) / 3600) * 10) / 10 
    AS truncated_hours,
    DATE_FORMAT(login_datetime, 'EEEE') AS weekday_name,
    CASE 
        WHEN DAYOFWEEK(login_datetime) NOT IN (1,7)
             AND HOUR(login_datetime) < 9
             AND ((UNIX_TIMESTAMP(logout_datetime) - UNIX_TIMESTAMP(login_datetime)) / 3600) >= 8
        THEN 'Disciplined' 
        WHEN DAYOFWEEK(login_datetime) NOT IN (1,7)
             AND HOUR(login_datetime) > 10
        THEN 'Late Comer'
        ELSE 'Poor Discipline'
    END AS status
FROM login_discipline;

login_hour,working_hours,truncated_hours,weekday_name,status
8,9.25,9.2,Wednesday,Disciplined
10,5.25,5.2,Wednesday,Poor Discipline
9,11.5,11.5,Wednesday,Poor Discipline
11,2.5,2.5,Wednesday,Late Comer


## QUESTION 5 – Absenteeism vs Performance Correlation

In [0]:
CREATE TABLE performance_tracker (
emp_id INT,
emp_name STRING,
login_ts TIMESTAMP,
logout_ts TIMESTAMP
);
INSERT INTO performance_tracker VALUES
(1,'Karthik','2025-01-09 09:00:00','2025-01-09 18:00:00'),
(2,'Karthik','2025-01-10 09:10:00','2025-01-10 17:50:00'),
(3,'Veena','2025-01-05 10:00:00','2025-01-05 15:00:00'),
(4,'Ravi','2025-01-14 09:00:00','2025-01-14 19:00:00'),
(5,'Anil','2025-01-03 11:00:00','2025-01-03 14:00:00');

num_affected_rows,num_inserted_rows
5,5


###### For each record identify whether the work date is within the last 7 days, determine if it is a weekday or weekend, calculate total working hours, apply FLOOR to the working hours, and use a CASE statement to classify as Consistent Performer if within last 7 days and weekday and hours are greater than or equal to 8, Irregular Performer if hours are less than 6, otherwise Absent or Old Record.

In [0]:
SELECT 
    CASE 
        WHEN DATE(login_ts) >= DATE_SUB(CURDATE(), 7) THEN 'Last 7 Days'
        ELSE 'Older'
    END AS recent_flag, 
    CASE 
        WHEN DAYOFWEEK(login_ts) IN (1,7) THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type, 
    (UNIX_TIMESTAMP(logout_ts) - UNIX_TIMESTAMP(login_ts)) / 3600 AS working_hours, 
    FLOOR((UNIX_TIMESTAMP(logout_ts) - UNIX_TIMESTAMP(login_ts)) / 3600) AS floored_hours,
    CASE 
        WHEN DATE(login_ts) >= DATE_SUB(CURDATE(), 7)
             AND DAYOFWEEK(login_ts) NOT IN (1,7)
             AND FLOOR((UNIX_TIMESTAMP(logout_ts) - UNIX_TIMESTAMP(login_ts)) / 3600) >= 8
        THEN 'Consistent Performer'   
        WHEN FLOOR((UNIX_TIMESTAMP(logout_ts) - UNIX_TIMESTAMP(login_ts)) / 3600) < 6
        THEN 'Irregular Performer'  
        ELSE 'Absent / Old Record'
    END AS status
FROM performance_tracker;

recent_flag,day_type,working_hours,floored_hours,status
Older,Weekday,9.0,9,Absent / Old Record
Older,Weekday,8.666666666666666,8,Absent / Old Record
Older,Weekend,5.0,5,Irregular Performer
Older,Weekday,10.0,10,Absent / Old Record
Older,Weekday,3.0,3,Irregular Performer
